In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from config.spark import spark


In [2]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,TimestampType,DoubleType

orders_schema = StructType([
    StructField("order_id",StringType()),
    StructField("customer_id",StringType()),
    StructField("order_status",StringType()),
    StructField("order_purchase_timestamp",TimestampType()),
    StructField("order_approved_at",TimestampType()),
    StructField("order_delivered_carrier_date",TimestampType()),
    StructField("order_delivered_customer_date",TimestampType()),
    StructField("order_estimated_delivery_date",TimestampType()),
])


In [3]:
orders_df = (
    spark.read.format('csv')
    .option('header','True')
   # .option('inferschema','True')
   .schema(orders_schema)
    .load('../landing/files/olist_orders_dataset.csv')
    
)

In [4]:
from pyspark.sql.functions import current_timestamp

orders_df = orders_df.withColumn("ingestion_time",current_timestamp())

In [5]:
orders_df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_time: timestamp (nullable = false)



In [8]:
orders_df.write.mode("overwrite").saveAsTable("bronze.orders")

In [ ]:
spark.sql(
    """
    select * from bronze.orders
    """
).show(100)